# Set up yearly turbine Parquet files

Set `YEARS` below and run all cells to convert the exported Excel workbooks. `TURBINE_IDS = None` converts every turbine sheet; use a list to select turbines. Each year's workbook is opened once and turbines are processed one at a time. Initial conversion can take several minutes per year.

Files are written under **Iteration2/Data_Parquets/<year>/Station_<ID>/**:
- `sensors.parquet`: numeric sensor features, sorted by timestamp, with a timestamp index. Alarm-only rows are excluded; missing sensor values remain missing.
- `alarms.parquet`: every recorded alarm event with its code, start and end time, including alarm-only timestamps.
- `source.json`: workbook details used to detect changes and skip unchanged conversions. Set `REBUILD = True` to force replacement.

Original Excel workbooks are preserved. Notebooks 06ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Å“09 continue to read Excel independently. This notebook prepares Parquet files for separate use; it does not change plotting notebooks. Dependencies (`pandas`, `openpyxl`, `pyarrow`) are already in `requirements.txt`.

For the shared plotting/training workflow: use notebook 02_parquet_setup/01_setup_yearly_parquets.ipynb for source Parquet, then notebook 03_preprocessing/01_preprocess_10min_parquets.ipynb for aligned interval Parquet. Notebooks 04–08 and 09 read notebook 03_preprocessing/01_preprocess_10min_parquets.ipynb outputs. notebook 01_data_export/02_optional_export_10min_alarm_scada.ipynb remains an optional separate Excel export.


In [ ]:
from pathlib import Path

WORKSPACE_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                       if (p / 'notebooks' / '01_data_export/01_export_yearly_turbine_scada.ipynb').is_file()), None)
if WORKSPACE_ROOT is None:
    raise FileNotFoundError('Open from Iteration2 or its notebooks folder.')

YEARS = [2016]  # Example: [2016, 2017, 2018]
TURBINE_IDS = None  # All turbine sheets; or ['2304510', '2304511'].
EXCEL_FOLDER = WORKSPACE_ROOT / 'Data' / 'Interim' / 'turbine_exports'
OUTPUT_FOLDER = WORKSPACE_ROOT / 'Data_Parquets'
REBUILD = False


In [ ]:
"""Persistent typed sensor and alarm caches for the feature notebooks."""
import json
from contextlib import nullcontext
import os
from pathlib import Path
from uuid import uuid4

import pandas as pd

CACHE_VERSION = 1


def load_turbine(workbook_path, turbine_id, cache_root, rebuild=False, excel_workbook=None):
    """Read a current Parquet cache, or build it once from the Excel turbine sheet."""
    workbook_path = Path(workbook_path).resolve()
    stat = workbook_path.stat()
    fingerprint = {"version": CACHE_VERSION, "source": str(workbook_path),
                   "size": stat.st_size, "mtime_ns": stat.st_mtime_ns}
    sheet = f"Station_{turbine_id}"
    if not str(turbine_id).isdigit():
        raise ValueError("TURBINE_ID must contain digits only.")
    year_folder = workbook_path.stem.removeprefix("turbines_")
    if len(year_folder) != 4 or not year_folder.isdigit():
        raise ValueError("Expected a yearly workbook named turbines_<year>.xlsx")
    folder = Path(cache_root) / year_folder / sheet
    sensors_path = folder / "sensors.parquet"
    alarms_path = folder / "alarms.parquet"
    manifest_path = folder / "source.json"
    try:
        current = json.loads(manifest_path.read_text(encoding="utf-8")) == fingerprint
    except (FileNotFoundError, ValueError):
        current = False
    if not rebuild and current and sensors_path.is_file() and alarms_path.is_file():
        print(f"Loading Parquet: {folder}")
        return pd.read_parquet(sensors_path), pd.read_parquet(alarms_path)

    print(f"Building Parquet cache from Excel ({sheet}); the first load can take minutes...")
    with (pd.ExcelFile(workbook_path, engine="openpyxl") if excel_workbook is None
          else nullcontext(excel_workbook)) as workbook:
        if sheet not in workbook.sheet_names:
            raise ValueError(f"{sheet} is unavailable. Sheets: {workbook.sheet_names}")
        data = pd.read_excel(workbook, sheet_name=sheet)
    features = [c for c in data.columns if c.startswith("wtc_")]
    if not features:
        raise ValueError("No wtc_ sensor features found in the sheet.")
    sensors = data.loc[data[features].notna().any(axis=1), ["TimeStamp", *features]].copy()
    sensors["TimeStamp"] = pd.to_datetime(sensors["TimeStamp"], errors="coerce").astype("datetime64[ns]")
    sensors[features] = sensors[features].apply(pd.to_numeric, errors="coerce")
    sensors = sensors.dropna(subset=["TimeStamp"]).sort_values("TimeStamp").set_index("TimeStamp")
    events = []
    if "AlarmEvents" in data.columns:
        for payload in data["AlarmEvents"].dropna():
            if str(payload).strip():
                events.extend(json.loads(payload))
    else:
        print("AlarmEvents column unavailable: no alarm overlay will be cached.")
    alarms = pd.DataFrame(events, columns=["Alarmcode", "TimeOn", "TimeOff"])
    alarms["Alarmcode"] = alarms["Alarmcode"].astype(str)
    for column in ["TimeOn", "TimeOff"]:
        alarms[column] = pd.to_datetime(alarms[column], errors="coerce").astype("datetime64[ns]")
    alarms = alarms.dropna(subset=["TimeOn"])
    if workbook_path.stat().st_mtime_ns != stat.st_mtime_ns or workbook_path.stat().st_size != stat.st_size:
        raise RuntimeError("The workbook changed during conversion. Rerun after the export completes.")
    folder.mkdir(parents=True, exist_ok=True)
    # Publish the manifest last so interrupted conversions are rebuilt next time.
    manifest_path.unlink(missing_ok=True)
    token = uuid4().hex
    temporary = [folder / f".{token}-{name}" for name in ["sensors.parquet", "alarms.parquet", "source.json"]]
    try:
        sensors.to_parquet(temporary[0], engine="pyarrow", compression="snappy")
        alarms.to_parquet(temporary[1], engine="pyarrow", compression="snappy", index=False)
        temporary[2].write_text(json.dumps(fingerprint, indent=2), encoding="utf-8")
        for staged, target in zip(temporary, [sensors_path, alarms_path, manifest_path], strict=False):
            os.replace(staged, target)
    finally:
        for staged in temporary:
            staged.unlink(missing_ok=True)
    print(f"Saved Parquet cache: {folder}")
    return sensors, alarms


In [ ]:
# Validate all requested years before starting conversion.
workbooks = [(year, EXCEL_FOLDER / f'turbines_{year}.xlsx') for year in YEARS]
missing = [str(path) for _, path in workbooks if not path.is_file()]
if missing:
    raise FileNotFoundError('Missing yearly workbooks: ' + ', '.join(missing))
summary = []
for year, path in workbooks:
    print(f'Year {year}: opening {path.name}...')
    with pd.ExcelFile(path, engine='openpyxl') as workbook:
        ids = ([s.removeprefix('Station_') for s in workbook.sheet_names if s.startswith('Station_')]
               if TURBINE_IDS is None else [str(t) for t in TURBINE_IDS])
        invalid = [t for t in ids if f'Station_{t}' not in workbook.sheet_names]
        if invalid:
            raise ValueError(f'{year}: turbine sheets unavailable: {invalid}')
        for turbine in ids:
            sensors, alarms = load_turbine(path, turbine, OUTPUT_FOLDER,
                                           rebuild=REBUILD, excel_workbook=workbook)
            summary.append({'Year': year, 'Turbine': turbine,
                            'Sensor rows': len(sensors), 'Alarm events': len(alarms),
                            'Folder': str(OUTPUT_FOLDER / str(year) / f'Station_{turbine}')})
            del sensors, alarms
print(f'Finished. Parquet files: {OUTPUT_FOLDER}')
display(pd.DataFrame(summary))


In [ ]:
# Example: inspect a generated turbine without opening Excel.
# Change these choices independently of the conversion settings.
EXAMPLE_YEAR = YEARS[0]
if not summary:
    raise ValueError('No turbines converted. Check the year/turbine settings.')
EXAMPLE_TURBINE = next(row['Turbine'] for row in summary if row['Year'] == EXAMPLE_YEAR)
parquet_folder = OUTPUT_FOLDER / str(EXAMPLE_YEAR) / f'Station_{EXAMPLE_TURBINE}'
sensors = pd.read_parquet(parquet_folder / 'sensors.parquet')
alarms = pd.read_parquet(parquet_folder / 'alarms.parquet')
display(sensors.head())
display(alarms.head())
